# Israeli Data Ingest — DSP + Split on Colab

End-to-end Colab notebook that turns the raw **SourcePool** (`.wav` + `.mid`
pairs already on Drive) into the training-ready tensor + split layout
consumed by `train_israeli.ipynb`.

**Run this notebook once per Israeli version.** Set `VERSION_NAME` in Cell 1,
run all cells, then switch `VERSION_NAME` and run again:
- `Israeli_Artists`  → `version_id 1` (88 songs, 13 artists)
- `Israeli_Military` → `version_id 2` (113 songs, 18 ensembles)

## Purpose
1. Run `preprocessing.derive_version` to DSP every song in
   `configs/version_<VERSION_NAME>.yaml` (including its 4 augmentations) into
   mel + piano-roll segment tensors.
2. Sanity-check the resulting tensors (shape + counts).
3. Separate **held-out songs** (D-set for inference) from the training pool.
4. Run `preprocessing.split_dataset` on the training pool → song-grouped
   train / val / test CSVs (a song + all its augmentations stay in one split).
5. Print a ready-to-paste summary.

## Data flow

```
G:/My Drive/MusicProject/SourcePool/<artist>/<album>/<song>/
    ├── <song>.wav, <song>.mid, metadata.json
    └── augmented/<song>_{ps+2,ps-2,ts0.9,ts1.1}.{wav,mid}

         ↓  (Cell 2)  python -m preprocessing.derive_version

G:/My Drive/MusicProject/versions/<VERSION_NAME>/
    ├── processed_data/<artist>/<album>/<song>/[<aug_tag>/]
    │       ├── mels/segment_NNNN.pt          [80, 430]
    │       └── piano_rolls/segment_NNNN.pt   [2, 128, 430]
    ├── manifest.csv                          ← all segments
    └── version_spec.copy.yaml

         ↓  (Cells 4–5)  held-out split + split_dataset

G:/My Drive/MusicProject/versions/<VERSION_NAME>/splits/
    ├── train.csv, val.csv, test.csv          ← training pool only
    └── held_out.csv                          ← D-set (inference)
```

## Pre-requisites
- SourcePool already populated (run `batch_ingest.py` locally first).
- `configs/version_<VERSION_NAME>.yaml` populated with songs + held_out_songs.
- Colab secret `GITHUB_TOKEN` set.


## Cell 1 — Environment setup

**What this does.** Mounts Drive, clones the repo (or pulls latest), installs
Colab requirements, and defines the path constants used by every later cell.

**Inputs.** Colab secret `GITHUB_TOKEN`; Drive mount at `/content/drive`.

**Outputs.** Global vars `REPO_DIR`, `SOURCE_POOL`, `OUT_VERSION_DIR`,
`VERSION_SPEC`, `MANIFEST_PATH`, `SPLITS_DIR`.

**Action required.** Authorise the Drive + GitHub prompts on first run.

**Runtime.** ~1 min.


In [ ]:
# ============================================================
# Cell 1 — Environment setup
# ============================================================
import os, sys, subprocess

print('[1/6] Mounting Google Drive ...', flush=True)
from google.colab import drive, userdata
drive.mount('/content/drive')
print('      Drive mounted.', flush=True)

print('[2/6] Reading GITHUB_TOKEN secret ...', flush=True)
_token = userdata.get('GITHUB_TOKEN')
assert _token, 'GITHUB_TOKEN Colab secret missing'
REPO_URL = f'https://{_token}@github.com/galgeva1/MusicProject.git'

REPO_DIR = '/content/MusicProject'
if not os.path.exists(REPO_DIR):
    print('[3/6] Cloning repo ...', flush=True)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('[3/6] Repo exists — resetting + pulling latest ...', flush=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'HEAD'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
print('      Repo ready.', flush=True)

%cd {REPO_DIR}
print('[4/6] Installing requirements (pip, quiet) ...', flush=True)
!pip install -q -r requirements_colab.txt
print('      Requirements installed.', flush=True)

sys.path.insert(0, REPO_DIR)

# ---- WHICH VERSION TO TENSORIZE --------------------------------------------
# Run this whole notebook once per Israeli version. Set VERSION_NAME, run all
# cells top-to-bottom, then come back here, switch to the next version, and
# re-run. Each version writes to its own versions/<VERSION_NAME>/ folder.
#   'Israeli_Artists'   -> version_id 1  (88 songs, 13 artists)
#   'Israeli_Military'  -> version_id 2  (113 songs, 18 ensembles)
VERSION_NAME = 'Israeli_Artists'

# ---- Centralised paths (single source of truth) ------------------------------
print(f'[5/6] Resolving paths for VERSION_NAME = {VERSION_NAME} ...', flush=True)
from paths import DrivePaths, assert_env, write_lock_file
assert_env(expect='/usr/')        # Colab runtime check
P = DrivePaths.colab(version_name=VERSION_NAME)
P.ensure_dirs()

print('[6/6] Verifying SourcePool + version spec exist ...', flush=True)
assert P.source_pool.is_dir(),  f'SourcePool missing: {P.source_pool}'
assert P.version_spec.is_file(), f'version_spec missing: {P.version_spec}'

print(P.summary())
print(f'VERSION_NAME  : {VERSION_NAME}')
print(f'version_spec  : {P.version_spec}')
print('Environment ready ✓', flush=True)


## Cell 2 — DSP via `derive_version`

**What this does.** Iterates over every (song, augmentation) pair listed in the
active version spec (`configs/version_<VERSION_NAME>.yaml`, set in Cell 1 —
e.g. `Israeli_Artists` = 88 songs, `Israeli_Military` = 113 songs), computes
mel-spectrograms + piano-rolls, segments them at 5 s, and writes tensors +
`manifest.csv` under `versions/<VERSION_NAME>/`. Idempotent — already-processed
(song, augmentation) pairs are skipped.

**Inputs.** `P.source_pool`, `P.version_spec` (both resolved from `VERSION_NAME`).

**Outputs.** `P.version_dir/processed_data/...`, `P.version_dir/manifest.csv`,
`P.version_dir/version_spec.copy.yaml`, `P.version_dir/derive_summary.json`.

**Action required.** None (just run; safe to re-run).

**Runtime.** Each song expands to 5 variants (orig + 4 augmentations), so
~88 songs × 5 ≈ 440 pairs for `Israeli_Artists` and ~113 × 5 ≈ 565 for
`Israeli_Military`. Expect roughly 1–2 h per version (mostly CPU DSP — GPU
type doesn't matter much). Re-running resumes where it left off.


In [ ]:
# ============================================================
# Cell 2 — Run preprocessing.derive_version
# ============================================================
# Streams per-song progress live: each song prints
#   [i/N] ▶ artist/album/song   then   orig/ps+2/... : <n> segments
# We force unbuffered output (-u + PYTHONUNBUFFERED) and read stdout
# line-by-line so the progress shows in real time, not in a burst at the end.
#
# Exit-code contract of derive_version._main():
#   0  -> every song succeeded
#   2  -> finished, but >=1 song failed (soft warning — the rest are valid)
#   1  -> could not start (bad source pool / unreadable spec)
# We treat 2 as a WARNING (list the failures, keep going) and only hard-stop
# on a real crash.
import json, os, subprocess, sys, time

cmd = [
    sys.executable, '-u', '-m', 'preprocessing.derive_version',
    '--source-pool',     str(P.source_pool),
    '--version-spec',    str(P.version_spec),
    '--out-version-dir', str(P.version_dir),
]
print('Running:', ' '.join(f'"{c}"' if ' ' in c else c for c in cmd))
print('-' * 60)

_env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8'}
_t0 = time.time()
proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line.rstrip(), flush=True)
proc.wait()

print('-' * 60)
print(f'derive_version finished in {(time.time() - _t0) / 60:.1f} min '
      f'(exit {proc.returncode})')

summary_path = P.version_dir / 'derive_summary.json'
DERIVE_SUMMARY = json.loads(summary_path.read_text()) if summary_path.exists() else {}

# Hard-stop only on a genuine crash (exit 1 = could not start; anything that
# isn't 0/2 is unexpected). Exit 2 means some songs failed but the version is
# still usable — surface the list and continue.
if proc.returncode not in (0, 2):
    raise AssertionError(f'derive_version crashed (exit {proc.returncode})')

failed = list(DERIVE_SUMMARY.get('songs_failed') or [])
if failed:
    print(f'\n⚠ {len(failed)} song(s) FAILED (rest are valid, '
          f'{DERIVE_SUMMARY.get("total_segments", "?")} segments written):')
    for f in failed:
        print(f'  ✗ {f}')
    print('\nDecide per song: fix the source data and re-run this cell '
          '(idempotent — only failed/new songs are reprocessed), or remove '
          'the song from the version spec if it is unrecoverable.')
else:
    print('✓ All songs processed with no failures.')

if DERIVE_SUMMARY:
    print('\nderive_summary.json:')
    for k, v in DERIVE_SUMMARY.items():
        if k == 'songs_ok':
            print(f'  {k}: {len(v)} songs')
        elif k == 'songs_failed':
            print(f'  {k}: {len(v)} songs')
        else:
            print(f'  {k}: {v}')


## Cell 3 — Tensor + manifest sanity check

**What this does.** Loads a random sample of mel + piano-roll tensors and
asserts shapes are `[80, 430]` and `[2, 128, 430]`. Also checks the
consolidated `manifest.csv` for required columns and that every row's
`version_id` matches the value declared in the version spec (1 for
`Israeli_Artists`, 2 for `Israeli_Military`).

**Inputs.** `OUT_VERSION_DIR/processed_data/...`, `MANIFEST_PATH`.

**Outputs.** Console report only (no files written).

**Action required.** Fail-stop the notebook if any error is reported.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 3 — Tensor + manifest sanity check
# ============================================================
import random, torch, pandas as pd, yaml

print('[1/4] Scanning processed_data for tensors ...', flush=True)
mel_files = sorted(P.processed_data_dir.rglob('**/mels/segment_*.pt'))
pr_files  = sorted(P.processed_data_dir.rglob('**/piano_rolls/segment_*.pt'))
print(f'Found {len(mel_files):>6} mel tensors')
print(f'Found {len(pr_files):>6} piano_roll tensors')
assert len(mel_files) == len(pr_files), 'mel/piano_roll count mismatch'
assert len(mel_files) > 0, 'No tensors found — did Cell 2 succeed?'

print('[2/4] Spot-checking tensor shapes ...', flush=True)
errors = []
sample = random.sample(mel_files, min(30, len(mel_files)))
for i, mel_path in enumerate(sample, start=1):
    pr_path = mel_path.parent.parent / 'piano_rolls' / mel_path.name
    mel = torch.load(mel_path, weights_only=True)
    pr  = torch.load(pr_path,  weights_only=True)
    if tuple(mel.shape) != (80, 430):
        errors.append(f'{mel_path}: mel shape {tuple(mel.shape)}')
    if tuple(pr.shape) != (2, 128, 430):
        errors.append(f'{pr_path}: piano_roll shape {tuple(pr.shape)}')
    print(f'  checked {i}/{len(sample)} sampled tensors', flush=True)

if errors:
    print(f'\n{len(errors)} SHAPE ERRORS:')
    for e in errors[:10]:
        print(f'  ✗ {e}')
    raise AssertionError('Shape errors found — stop here')
print('✓ All sampled tensors have expected shapes')

# ---- manifest.csv sanity ----
print('[3/4] Validating manifest.csv columns + version_id ...', flush=True)
df = pd.read_csv(P.manifest_csv)
print(f'\nmanifest.csv: {len(df)} rows, {df["song_id"].nunique()} unique song_ids')
required = {'segment_path', 'score_path', 'version_id', 'song_id',
            'artist', 'album', 'song_name', 'aug_tag', 'segment_idx', 'duration_s'}
missing_cols = required - set(df.columns)
assert not missing_cols, f'manifest missing columns: {missing_cols}'

# version_id is taken from the spec (1 = Israeli_Artists, 2 = Israeli_Military),
# so this check works for whichever VERSION_NAME you set in Cell 1.
EXPECTED_VID = yaml.safe_load(P.version_spec.read_text(encoding='utf-8'))['version_id']
assert (df['version_id'] == EXPECTED_VID).all(), \
    f'manifest has rows with version_id != {EXPECTED_VID}'
print(f'✓ All rows have version_id == {EXPECTED_VID}')

# ---- on-disk completeness: every referenced tensor must exist ----
# Guards against interrupted .pt writes (Drive/FUSE flakiness) where the
# per-song manifest was written but some segments never landed.
print('[4/4] Verifying every manifest tensor exists on disk ...', flush=True)
missing_files = [s for s in df['segment_path'] if not (P.version_dir / s).exists()]
missing_files += [s for s in df['score_path'] if not (P.version_dir / s).exists()]
if missing_files:
    print(f'\n{len(missing_files)} MISSING TENSORS (first 10):')
    for s in missing_files[:10]:
        print(f'  ✗ {s}')
    raise AssertionError(
        f'{len(missing_files)} tensors referenced by manifest are missing on disk. '
        'Delete the affected per-pair folder(s) and re-run Cell 2 to regenerate.')
print(f'✓ All {len(df)} manifest tensors present on disk')

print('\nSegments per aug_tag:')
print(df.groupby('aug_tag').size().to_string())

dur_h = df['duration_s'].sum() / 3600
print(f'\nTotal audio duration in version: {dur_h:.2f} h')


## Cell 4 — Separate held-out songs (D-set)

**What this does.** Reads `held_out_songs` from the version spec and splits
the manifest into two CSVs:
- `train_pool_manifest.csv` — fed into `split_dataset` next.
- `held_out.csv` — never seen during training; used at inference time by
  `run_spec_Israeli_Shalom_Arik.yaml`.

Held-out filtering uses **`song_name`** so it covers the original recording
*and* all 4 augmentations of that song (no leakage from `_ps+2` etc.).

**Inputs.** `MANIFEST_PATH`, `VERSION_SPEC`.

**Outputs.** `OUT_VERSION_DIR/train_pool_manifest.csv`,
`OUT_VERSION_DIR/held_out.csv`.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 4 — Separate held-out songs from training pool
# ============================================================
import yaml, pandas as pd

spec = yaml.safe_load(P.version_spec.read_text(encoding='utf-8'))
held_out_names = list(spec.get('held_out_songs') or [])
print(f'held_out_songs ({len(held_out_names)}):')
for n in held_out_names:
    print(f'  - {n}')

df = pd.read_csv(P.manifest_csv)

# Defensive: ensure every held-out name is present in the manifest
manifest_song_names = set(df['song_name'].unique())
missing = [n for n in held_out_names if n not in manifest_song_names]
assert not missing, f'held_out song_names not in manifest: {missing}'

held_mask  = df['song_name'].isin(held_out_names)
train_pool = df[~held_mask].copy()
held_out   = df[ held_mask].copy()

train_pool.to_csv(P.train_pool_manifest, index=False)
held_out.to_csv  (P.held_out_manifest,   index=False)

print(f'\nTrain pool : {len(train_pool):>6} segments  ({train_pool["song_name"].nunique()} unique songs)')
print(f'             {train_pool["duration_s"].sum()/3600:.2f} h audio')
print(f'Held out   : {len(held_out):>6} segments  ({held_out["song_name"].nunique()} unique songs)')
print(f'             {held_out["duration_s"].sum()/3600:.2f} h audio')
print(f'\nWritten:')
print(f'  {P.train_pool_manifest}')
print(f'  {P.held_out_manifest}')


## Cell 5 — Song-grouped train / val / test split

**What this does.** Calls `preprocessing.split_dataset.split_dataset` on the
`train_pool_manifest.csv`. Splits are stratified by `song_id` (so a song
plus its augmentations stay in the same split — no leakage). 80 / 10 / 10
ratio, seed = 42.

**Inputs.** `train_pool_manifest.csv`.

**Outputs.** `SPLITS_DIR/{train,val,test}.csv`.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 5 — Song-grouped split: train / val / test
# Group by base song (artist, album, song_name) so a song AND all of its
# augmented variants (orig, ps+2, ps-2, ts0.9, ts1.1) always land in the
# SAME split — no augmentation leakage across train/val/test.
# ============================================================
from preprocessing.split_dataset import split_dataset

SPLIT_RESULTS = split_dataset(
    manifest_path = P.train_pool_manifest,
    out_dir       = P.splits_dir,
    train_frac    = 0.80,
    val_frac      = 0.10,
    test_frac     = 0.10,
    seed          = 42,
    group_by      = ['artist', 'album', 'song_name'],
)

print('\nWritten:')
for split, info in SPLIT_RESULTS.items():
    print(f'  {split:5s}  {info["segments"]:>5} segments  {info["songs"]:>3} songs  → {info["path"]}')

# Hard verify: no base song appears in more than one split
import pandas as pd
_seen = {}
for split in ('train', 'val', 'test'):
    _df = pd.read_csv(P.splits_dir / f'{split}.csv')
    _keys = set(_df[['artist', 'album', 'song_name']].astype(str).agg('|'.join, axis=1))
    for k in _keys:
        assert k not in _seen, f'LEAK: base song {k!r} in both {_seen[k]} and {split}'
        _seen[k] = split
print(f'OK: {len(_seen)} base songs, each confined to a single split (no augmentation leakage)')


## Cell 6 — Early visualizations (data quality preview)

**What this does.** Runs the same TA-grading visualizations that
`train_israeli.ipynb` produces later — but here, against the freshly built
manifest. Gives you an early visual gate so you don't spend GPU on bad data:
mel grid, piano-roll grid, segment-length histogram, MFCC similarity heatmap,
and dataset-stats panel. PNGs go next to the splits.

**Inputs.** `P.train_pool_manifest`, `P.processed_data_dir`.

**Outputs.** `P.version_dir/_viz/*.png`.

**Action required.** Eyeball the images. If anything looks off, stop here.

**Runtime.** ~30 sec.


In [ ]:
# ============================================================
# Cell 6 — Early visualizations
# ============================================================
import pandas as pd
from preprocessing.dataset_visualizations import (
    plot_mel_grid,
    plot_piano_roll_grid,
    plot_segment_length_histogram,
    plot_mfcc_similarity_heatmap,
    plot_dataset_stats_panel,
)

VIZ_DIR = P.version_dir / '_viz'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# These viz functions accept a DataFrame and a manifest-root path. The
# segment_path values in the manifest already include the `processed_data/`
# prefix, so the root must be `version_dir` (NOT processed_data_dir, which
# would double the prefix).
viz_df   = pd.read_csv(P.train_pool_manifest)
viz_root = P.version_dir

print('[1/5] dataset_stats panel ...', flush=True)
plot_dataset_stats_panel(viz_df, save_path=str(VIZ_DIR / 'dataset_stats.png'))
print('[2/5] mel grid ...', flush=True)
plot_mel_grid           (viz_df, viz_root, n=8, save_path=str(VIZ_DIR / 'mel_grid.png'))
print('[3/5] piano-roll grid ...', flush=True)
plot_piano_roll_grid    (viz_df, viz_root, n=8, save_path=str(VIZ_DIR / 'piano_roll_grid.png'))
print('[4/5] segment-length histogram ...', flush=True)
plot_segment_length_histogram(viz_df, viz_root, save_path=str(VIZ_DIR / 'segment_length_hist.png'))
print('[5/5] MFCC similarity heatmap ...', flush=True)
plot_mfcc_similarity_heatmap (viz_df, viz_root, save_path=str(VIZ_DIR / 'mfcc_heatmap.png'))

print(f'\nViz written under: {VIZ_DIR}')
for p in sorted(VIZ_DIR.glob('*.png')):
    print(f'  - {p.name}')


## Cell 7 — Reproducibility lock file

**What this does.** Writes `version_Israeli_Shalom_Arik.lock.json` next to the
processed data. The lock pins this dataset to:
- git commit + dirty flag
- SHA256 of `configs/version_Israeli_Shalom_Arik.yaml` + `configs/default.yaml`
- segment / hour totals (from `derive_summary.json`)

`train_israeli.ipynb` re-checks this lock at startup and refuses to train
if the commit or configs drift, eliminating the "trained on the wrong data"
class of bug.

**Inputs.** Git repo state, version spec, default config, derive summary.

**Outputs.** `P.lock_file`.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 7 — Write reproducibility lock file
# ============================================================
import json

lock_path = write_lock_file(
    P,
    derive_summary=DERIVE_SUMMARY if 'DERIVE_SUMMARY' in dir() else None,
    extra={
        'splits': {k: {'segments': v['segments'], 'songs': v['songs']}
                   for k, v in SPLIT_RESULTS.items()},
        'held_out_songs': held_out_names,
        'held_out_segments': int(len(held_out)),
        'train_pool_segments': int(len(train_pool)),
    },
)
print(f'Lock file written: {lock_path}')
print('-' * 60)
print(json.dumps(json.loads(lock_path.read_text(encoding='utf-8')), indent=2))


## Cell 8 — Final summary

**What this does.** Prints a compact summary you can paste into
`ENGINEERING_DECISIONS.md`. Confirms the version is ready for training —
next step is to open `train_israeli.ipynb`.

**Inputs.** All artifacts produced by Cells 2–7.

**Outputs.** Console only.

**Action required.** Open `train_israeli.ipynb` next.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 8 — Final summary
# ============================================================
print('=' * 60)
print(f'VERSION READY: {P.version_name}')
print('=' * 60)
print(f'Lock file       : {P.lock_file}')
print(f'Manifest (all)  : {P.manifest_csv}  ({len(df)} segments)')
print(f'Train pool      : {P.train_pool_manifest}  ({len(train_pool)} segments)')
print(f'Held out        : {P.held_out_manifest}  ({len(held_out)} segments)')
for split, info in SPLIT_RESULTS.items():
    print(f'  {split:5s}        : {info["path"]}  ({info["segments"]} segments)')
print(f'Visualizations  : {P.version_dir / "_viz"}')
print()
print('Next: open colab/train_israeli.ipynb and run.')
print('=' * 60)
